In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the existing results
df = pd.read_csv('ldh_kappa_perturbation_results.csv')
print("Existing data loaded successfully")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nKappa values in dataset: {df['kappa'].unique()}")
print(f"\nData summary by kappa:")
print(df.groupby('kappa')['difference_pct'].agg(['mean', 'std', 'count']))


Existing data loaded successfully
Shape: (40, 7)

Columns: ['t', 'magnitude', 'red_S2_pct', 'red_S3_pct', 'difference_pct', 'kappa', 'perturbation']

Kappa values in dataset: [0.33408 0.23408]

Data summary by kappa:
 mean std count
kappa 
0.23408 14.982321 15.747325 20
0.33408 11.339830 14.602648 20


In [2]:

# Analysis Plan:
# 1. Implement the L_DH Dirichlet polynomial function with adjustable kappa
# 2. Load omega values for N=10^6 for omega-class decomposition
# 3. For kappa=0.15 and kappa=0.20:
# a. Perform coarse grid search (10,000 points) with Numba-accelerated summation over t in [1e6, 2e6]
# b. Find top 10 peaks from coarse search
# c. Refine each peak using Kahan summation + scipy.optimize.minimize_scalar
# d. At each refined peak, compute omega-class decomposition
# e. Perform causal perturbation: multiply S2 by e^(i*pi) and S3 by e^(i*pi) separately
# f. Calculate reduction percentages and differences
# 4. Combine with existing data and create final plot

# First, let's check the omega values file
import pickle

print("Loading omega values...")
with open('omega_values_N1e6.pkl', 'rb') as f:
 omega_values = pickle.load(f)
 
print(f"Omega values shape: {omega_values.shape}")
print(f"First 10 omega values (for n=1 to 10): {omega_values[:10]}")
print(f"Note: 0-based indexing, omega_values[i] = Omega(i+1)")


Loading omega values...
Omega values shape: (1000000,)
First 10 omega values (for n=1 to 10): [0 1 1 2 1 2 1 3 2 2]
Note: 0-based indexing, omega_values[i] = Omega(i+1)


In [3]:

# Implement the L_DH function
# L_DH(s) = zeta(s) * L(s, chi_DH)
# where chi_DH is the Dirichlet character mod 5 with chi(1)=1, chi(2)=i, chi(3)=i, chi(4)=-1
# For the Dirichlet polynomial: D_F(t; N) = sum_{n=1}^N a_n / n^(1/2 + it)
# where a_n = sum_{d|n} chi_DH(d) for L_DH

# However, the dataset description mentions kappa-perturbed versions of L_DH
# From the context, kappa seems to be a parameter that modifies the function
# Let me check what the canonical kappa value is

# The existing data has kappa ~0.234 and ~0.334
# These appear to be canonical ± 0.05
# So canonical kappa = (0.23408 + 0.33408)/2 = 0.28408

canonical_kappa = 0.28408
print(f"Inferred canonical kappa: {canonical_kappa}")

# The kappa perturbation likely modifies the character or the coefficients
# A common approach is to use: chi_kappa(n) = chi(n) * n^(kappa)
# or modify the summation exponent: n^(sigma + it) where sigma = 1/2 + kappa_adjustment

# Let me implement a flexible L_DH coefficient generator
def compute_ldh_coefficients(N, kappa=0):
 """
 Compute coefficients for the kappa-perturbed L_DH function.
 Based on standard L_DH with Dirichlet character mod 5.
 Kappa perturbation modifies the power: n^(kappa) scaling.
 """
 # Dirichlet character mod 5: chi(1)=1, chi(2)=i, chi(3)=i, chi(4)=-1, chi(0)=0
 chi_values = [0, 1, 1j, 1j, -1] # chi[n % 5]
 
 # For L_DH: a_n = sum_{d|n} chi(d)
 # This is the Dirichlet convolution of chi with 1
 a_n = np.zeros(N + 1, dtype=complex)
 
 for n in range(1, N + 1):
 # Sum over divisors of n
 divisor_sum = 0
 d = 1
 while d * d <= n:
 if n % d == 0:
 divisor_sum += chi_values[d % 5]
 if d != n // d:
 divisor_sum += chi_values[(n // d) % 5]
 d += 1
 a_n[n] = divisor_sum
 
 # Apply kappa perturbation: multiply by n^kappa
 if kappa != 0:
 n_array = np.arange(N + 1, dtype=float)
 n_array[0] = 1 # Avoid 0^kappa issue
 a_n *= n_array**kappa
 
 return a_n

print("Testing L_DH coefficient computation...")
a_test = compute_ldh_coefficients(20, kappa=0)
print(f"First 20 L_DH coefficients (kappa=0): {a_test[1:21]}")


Inferred canonical kappa: 0.28408
Testing L_DH coefficient computation...
First 20 L_DH coefficients (kappa=0): [1.+0.j 1.+1.j 1.+1.j 0.+1.j 1.+0.j 2.+2.j 1.+1.j 0.+2.j 0.+1.j 1.+1.j
 2.+0.j 1.+3.j 1.+1.j 0.+2.j 1.+1.j 1.+2.j 1.+1.j 1.+3.j 0.+0.j 0.+1.j]


In [4]:

# Now implement the Numba-accelerated Dirichlet polynomial summation
from numba import jit

@jit(nopython=True)
def dirichlet_sum_numba(coeffs_real, coeffs_imag, t, N):
 """
 Fast summation of Dirichlet polynomial using Numba.
 D_F(t; N) = sum_{n=1}^N a_n / n^(1/2 + it)
 """
 result_real = 0.0
 result_imag = 0.0
 
 for n in range(1, N + 1):
 # n^(-1/2 - it) = n^(-1/2) * n^(-it) = n^(-1/2) * exp(-it * log(n))
 n_sqrt = np.sqrt(float(n))
 log_n = np.log(float(n))
 angle = -t * log_n
 
 # exp(i*angle) = cos(angle) + i*sin(angle)
 cos_angle = np.cos(angle)
 sin_angle = np.sin(angle)
 
 # (a + ib) * (c + id) = (ac - bd) + i(ad + bc)
 # a_n / n^(1/2 + it) = a_n * n^(-1/2) * exp(-it*log(n))
 factor_real = cos_angle / n_sqrt
 factor_imag = sin_angle / n_sqrt
 
 term_real = coeffs_real[n] * factor_real - coeffs_imag[n] * factor_imag
 term_imag = coeffs_real[n] * factor_imag + coeffs_imag[n] * factor_real
 
 result_real += term_real
 result_imag += term_imag
 
 return result_real, result_imag

def evaluate_dirichlet_numba(coeffs, t, N):
 """Wrapper for Numba function."""
 coeffs_real = np.real(coeffs).astype(np.float64)
 coeffs_imag = np.imag(coeffs).astype(np.float64)
 real, imag = dirichlet_sum_numba(coeffs_real, coeffs_imag, t, N)
 return complex(real, imag)

print("Testing Numba summation...")
N_test = 1000
t_test = 1000.0
a_test = compute_ldh_coefficients(N_test, kappa=0.15)
result = evaluate_dirichlet_numba(a_test, t_test, N_test)
print(f"Test evaluation at t={t_test}, N={N_test}: {result}")
print(f"Magnitude: {abs(result):.4f}")


Testing Numba summation...


Test evaluation at t=1000.0, N=1000: (-0.021688411055407433+5.152253160786263j)
Magnitude: 5.1523


In [5]:

# Now implement Kahan summation for high-precision refinement
def kahan_sum_dirichlet(coeffs, t, N):
 """
 High-precision Kahan summation for Dirichlet polynomial.
 Returns the complex value D_F(t; N).
 """
 result_real = 0.0
 result_imag = 0.0
 compensation_real = 0.0
 compensation_imag = 0.0
 
 for n in range(1, N + 1):
 n_sqrt = np.sqrt(float(n))
 log_n = np.log(float(n))
 angle = -t * log_n
 
 cos_angle = np.cos(angle)
 sin_angle = np.sin(angle)
 
 factor_real = cos_angle / n_sqrt
 factor_imag = sin_angle / n_sqrt
 
 a_real = np.real(coeffs[n])
 a_imag = np.imag(coeffs[n])
 
 term_real = a_real * factor_real - a_imag * factor_imag
 term_imag = a_real * factor_imag + a_imag * factor_real
 
 # Kahan summation for real part
 y_real = term_real - compensation_real
 t_real = result_real + y_real
 compensation_real = (t_real - result_real) - y_real
 result_real = t_real
 
 # Kahan summation for imaginary part
 y_imag = term_imag - compensation_imag
 t_imag = result_imag + y_imag
 compensation_imag = (t_imag - result_imag) - y_imag
 result_imag = t_imag
 
 return complex(result_real, result_imag)

print("Testing Kahan summation...")
result_kahan = kahan_sum_dirichlet(a_test, t_test, N_test)
print(f"Kahan result: {result_kahan}")
print(f"Numba result: {result}")
print(f"Difference: {abs(result_kahan - result):.10e}")


Testing Kahan summation...
Kahan result: (-0.02168841105285331+5.152253160789048j)
Numba result: (-0.021688411055407433+5.152253160786263j)
Difference: 3.7784454116e-12


In [6]:

# Now compute coefficients for the full N=10^6 for both new kappa values
print("Computing L_DH coefficients for N=10^6...")
print("This may take a moment...")

N = 1000000

print("\nComputing for kappa=0.15...")
coeffs_015 = compute_ldh_coefficients(N, kappa=0.15)
print(f"Coefficients computed. Sample: {coeffs_015[100:105]}")

print("\nComputing for kappa=0.20...")
coeffs_020 = compute_ldh_coefficients(N, kappa=0.20)
print(f"Coefficients computed. Sample: {coeffs_020[100:105]}")

print("\nCoefficients ready for analysis.")


Computing L_DH coefficients for N=10^6...
This may take a moment...

Computing for kappa=0.15...


Coefficients computed. Sample: [0. +1.99526231j 3.99648513+0.j 4.00239568+8.00479137j
 2.00412859+2.00412859j 0. +8.02814101j]

Computing for kappa=0.20...


Coefficients computed. Sample: [0. +2.51188643j 5.03378046 +0.j 5.0437091 +10.08741819j
 2.52678008 +2.52678008j 0. +10.12667003j]

Coefficients ready for analysis.


In [7]:

# Perform coarse grid search for kappa=0.15
print("=" * 70)
print("KAPPA = 0.15: COARSE GRID SEARCH")
print("=" * 70)

kappa = 0.15
t_min = 1e6
t_max = 2e6
n_points = 10000

t_grid = np.linspace(t_min, t_max, n_points)

print(f"Evaluating {n_points} points in range [{t_min:.0f}, {t_max:.0f}]...")
print("Using Numba-accelerated summation...")

magnitudes_015 = np.zeros(n_points)

# Pre-split coefficients for Numba
coeffs_real_015 = np.real(coeffs_015).astype(np.float64)
coeffs_imag_015 = np.imag(coeffs_015).astype(np.float64)

import time
start = time.time()

for i, t in enumerate(t_grid):
 real, imag = dirichlet_sum_numba(coeffs_real_015, coeffs_imag_015, t, N)
 magnitudes_015[i] = np.sqrt(real**2 + imag**2)
 
 if (i + 1) % 2000 == 0:
 elapsed = time.time() - start
 print(f" Progress: {i+1}/{n_points} ({(i+1)/n_points*100:.1f}%) - {elapsed:.1f}s elapsed")

elapsed = time.time() - start
print(f"\nGrid search completed in {elapsed:.1f} seconds")
print(f"Max magnitude found: {magnitudes_015.max():.4f} at t={t_grid[magnitudes_015.argmax()]:.2f}")


KAPPA = 0.15: COARSE GRID SEARCH
Evaluating 10000 points in range [1000000, 2000000]...
Using Numba-accelerated summation...


 Progress: 2000/10000 (20.0%) - 79.2s elapsed


 Progress: 4000/10000 (40.0%) - 159.8s elapsed


 Progress: 6000/10000 (60.0%) - 240.5s elapsed


 Progress: 8000/10000 (80.0%) - 321.9s elapsed


 Progress: 10000/10000 (100.0%) - 403.1s elapsed

Grid search completed in 403.1 seconds
Max magnitude found: 1652.9196 at t=1508650.87


In [8]:

# Find top 10 peaks from coarse grid
print("Finding top 10 peaks from coarse grid...")

# Sort by magnitude and get top 10
top_indices = np.argsort(magnitudes_015)[-10:][::-1]
top_t_coarse_015 = t_grid[top_indices]
top_mag_coarse_015 = magnitudes_015[top_indices]

print("\nTop 10 peaks (coarse):")
for i, (t, mag) in enumerate(zip(top_t_coarse_015, top_mag_coarse_015)):
 print(f" {i+1}. t = {t:12.2f}, |D| = {mag:8.4f}")


Finding top 10 peaks from coarse grid...

Top 10 peaks (coarse):
 1. t = 1508650.87, |D| = 1652.9196
 2. t = 1444744.47, |D| = 1534.4017
 3. t = 1015701.57, |D| = 1300.8168
 4. t = 1034203.42, |D| = 1264.7615
 5. t = 1067506.75, |D| = 1245.3902
 6. t = 1523652.37, |D| = 1174.9981
 7. t = 1064606.46, |D| = 1162.9292
 8. t = 1937493.75, |D| = 1094.0911
 9. t = 1587558.76, |D| = 1056.2383
 10. t = 1067706.77, |D| = 1028.5432


In [9]:

# Refine peaks using scipy.optimize with Kahan summation
from scipy.optimize import minimize_scalar

print("Refining peaks using Kahan summation and scipy optimization...")
print("=" * 70)

refined_peaks_015 = []

for i, t_coarse in enumerate(top_t_coarse_015):
 print(f"\nRefining peak {i+1}/10 (initial t = {t_coarse:.2f})...")
 
 # Define objective function (negative magnitude for minimization)
 def neg_magnitude(t):
 result = kahan_sum_dirichlet(coeffs_015, t, N)
 return -abs(result)
 
 # Refine around the coarse peak (search window of ±100)
 result = minimize_scalar(neg_magnitude, 
 bounds=(t_coarse - 100, t_coarse + 100),
 method='bounded',
 options={'xatol': 1e-6})
 
 t_refined = result.x
 mag_refined = -result.fun
 
 print(f" Refined: t = {t_refined:.6f}, |D| = {mag_refined:.6f}")
 refined_peaks_015.append((t_refined, mag_refined))

print("\n" + "=" * 70)
print("All peaks refined for kappa=0.15")


Refining peaks using Kahan summation and scipy optimization...

Refining peak 1/10 (initial t = 1508650.87)...


 Refined: t = 1508674.411895, |D| = 48.319580

Refining peak 2/10 (initial t = 1444744.47)...


 Refined: t = 1444744.447084, |D| = 1537.093416

Refining peak 3/10 (initial t = 1015701.57)...


 Refined: t = 1015656.649601, |D| = 564.394202

Refining peak 4/10 (initial t = 1034203.42)...


 Refined: t = 1034257.741311, |D| = 103.693477

Refining peak 5/10 (initial t = 1067506.75)...


...

 Refined: t = 1523652.461171, |D| = 1203.114773

Refining peak 7/10 (initial t = 1064606.46)...


 Refined: t = 1064606.504517, |D| = 1172.816973

Refining peak 8/10 (initial t = 1937493.75)...


 Refined: t = 1937545.168707, |D| = 84.300444

Refining peak 9/10 (initial t = 1587558.76)...


 Refined: t = 1587558.766994, |D| = 1056.045782

Refining peak 10/10 (initial t = 1067706.77)...


 Refined: t = 1067695.505218, |D| = 44.551331

All peaks refined for kappa=0.15


In [10]:

# The refinement found some peaks with very different magnitudes than coarse search
# This is concerning - let me check what happened
print("Comparison of coarse vs refined magnitudes:")
print("\n{:>3} {:>15} {:>12} {:>15} {:>12} {:>10}".format(
 "#", "t_coarse", "|D|_coarse", "t_refined", "|D|_refined", "Change %"))
print("-" * 80)

for i in range(10):
 t_c = top_t_coarse_015[i]
 mag_c = top_mag_coarse_015[i]
 t_r, mag_r = refined_peaks_015[i]
 change_pct = (mag_r - mag_c) / mag_c * 100
 print(f"{i+1:3d} {t_c:15.2f} {mag_c:12.4f} {t_r:15.6f} {mag_r:12.6f} {change_pct:10.2f}%")

# Many peaks show massive magnitude drops - this suggests the coarse grid
# hit oscillatory behavior rather than true local maxima
# I need to re-sort by the refined magnitudes and keep only the top 10 actual peaks


Comparison of coarse vs refined magnitudes:

 # t_coarse |D|_coarse t_refined |D|_refined Change %
--------------------------------------------------------------------------------
 1 1508650.87 1652.9196 1508674.411895 48.319580 -97.08%
 2 1444744.47 1534.4017 1444744.447084 1537.093416 0.18%
 3 1015701.57 1300.8168 1015656.649601 564.394202 -56.61%
 4 1034203.42 1264.7615 1034257.741311 103.693477 -91.80%
 5 1067506.75 1245.3902 1067454.576811 147.082714 -88.19%
 6 1523652.37 1174.9981 1523652.461171 1203.114773 2.39%
 7 1064606.46 1162.9292 1064606.504517 1172.816973 0.85%
 8 1937493.75 1094.0911 1937545.168707 84.300444 -92.29%
 9 1587558.76 1056.2383 1587558.766994 1056.045782 -0.02%
 10 1067706.77 1028.5432 1067695.505218 44.551331 -95.67%


In [11]:

# Sort refined peaks by magnitude and keep top 10
print("Re-sorting by refined magnitude to get true top 10 peaks...")

refined_peaks_015_sorted = sorted(refined_peaks_015, key=lambda x: x[1], reverse=True)
top_10_peaks_015 = refined_peaks_015_sorted[:10]

print("\nTop 10 peaks after refinement (kappa=0.15):")
for i, (t, mag) in enumerate(top_10_peaks_015):
 print(f" {i+1}. t = {t:15.6f}, |D| = {mag:12.6f}")

# Check if we need more peaks - if several refined peaks are not in top 10,
# we should search more of the coarse grid
print(f"\nNumber of refined peaks with |D| > {top_10_peaks_015[-1][1]:.2f}: {len([p for p in refined_peaks_015 if p[1] > top_10_peaks_015[-1][1]])}")

# This is problematic - we only found 10 peaks and some collapsed to low magnitudes
# Let me expand the coarse search to look at more candidates


Re-sorting by refined magnitude to get true top 10 peaks...

Top 10 peaks after refinement (kappa=0.15):
 1. t = 1444744.447084, |D| = 1537.093416
 2. t = 1523652.461171, |D| = 1203.114773
 3. t = 1064606.504517, |D| = 1172.816973
 4. t = 1587558.766994, |D| = 1056.045782
 5. t = 1015656.649601, |D| = 564.394202
 6. t = 1067454.576811, |D| = 147.082714
 7. t = 1034257.741311, |D| = 103.693477
 8. t = 1937545.168707, |D| = 84.300444
 9. t = 1508674.411895, |D| = 48.319580
 10. t = 1067695.505218, |D| = 44.551331

Number of refined peaks with |D| > 44.55: 9


In [12]:

# The refinement is too slow. Let me use a faster approach:
# 1. Use the coarse grid magnitudes directly to select peaks
# 2. Apply a simple local maximum filter to avoid oscillation artifacts
# 3. Then refine only the true local maxima

from scipy.signal import find_peaks

print("Using peak detection on coarse grid...")

# Find local maxima with minimum distance between peaks
peaks_indices, properties = find_peaks(magnitudes_015, distance=10, height=200)

# Sort by height and take top 15 for refinement
top_peak_heights = properties['peak_heights']
sorted_peak_indices = np.argsort(top_peak_heights)[-15:][::-1]
selected_peaks = peaks_indices[sorted_peak_indices]

print(f"Found {len(selected_peaks)} candidate peaks for refinement")
print("\nTop 15 candidate peaks (coarse):")
for i, idx in enumerate(selected_peaks):
 print(f" {i+1}. t = {t_grid[idx]:12.2f}, |D| = {magnitudes_015[idx]:8.4f}")


Using peak detection on coarse grid...
Found 15 candidate peaks for refinement

Top 15 candidate peaks (coarse):
 1. t = 1508650.87, |D| = 1652.9196
 2. t = 1444744.47, |D| = 1534.4017
 3. t = 1015701.57, |D| = 1300.8168
 4. t = 1034203.42, |D| = 1264.7615
 5. t = 1067506.75, |D| = 1245.3902
 6. t = 1523652.37, |D| = 1174.9981
 7. t = 1064606.46, |D| = 1162.9292
 8. t = 1937493.75, |D| = 1094.0911
 9. t = 1587558.76, |D| = 1056.2383
 10. t = 1672867.29, |D| = 981.7130
 11. t = 1030903.09, |D| = 978.9166
 12. t = 1493449.34, |D| = 946.6890
 13. t = 1000700.07, |D| = 898.6564
 14. t = 1873587.36, |D| = 884.4011
 15. t = 1639363.94, |D| = 879.9595


In [13]:

# Refine these 15 peaks quickly (should take ~10-15 min)
print("Refining 15 candidate peaks for kappa=0.15...")
print("=" * 70)

refined_015_v2 = []
start = time.time()

for i, idx in enumerate(selected_peaks):
 t_coarse = t_grid[idx]
 
 if (i + 1) % 5 == 0:
 elapsed = time.time() - start
 print(f"Progress: {i+1}/15 peaks refined ({elapsed:.1f}s elapsed)")
 
 def neg_magnitude(t):
 result = kahan_sum_dirichlet(coeffs_015, t, N)
 return -abs(result)
 
 result = minimize_scalar(neg_magnitude, 
 bounds=(t_coarse - 100, t_coarse + 100),
 method='bounded',
 options={'xatol': 1e-6, 'maxiter': 50})
 
 t_refined = result.x
 mag_refined = -result.fun
 refined_015_v2.append((t_refined, mag_refined))

elapsed = time.time() - start
print(f"Refinement completed in {elapsed:.1f} seconds")

# Sort and get top 10
refined_015_v2_sorted = sorted(refined_015_v2, key=lambda x: x[1], reverse=True)
top_10_peaks_015_final = refined_015_v2_sorted[:10]

print("\n" + "=" * 70)
print("FINAL Top 10 peaks for kappa=0.15:")
for i, (t, mag) in enumerate(top_10_peaks_015_final):
 print(f" {i+1}. t = {t:15.6f}, |D| = {mag:12.6f}")


TimeoutError: Code execution timed out after 878 seconds